# Function_Calling

In [ ]:
"""
================================================================================
📌 目标：在 langchain_classic（AgentExecutor 体系）中，为你的 “Search + Calculator” 金融 Agent
      ✅ 加入 Memory（对话记忆）
      ✅ 并补充“详细、丰富、易读”的注释，方便学习/维护

⚠️ 说明：
- 你当前代码使用的是 langchain_classic（旧 Agents 体系的兼容包）
- Memory 的关键是：prompt 里要有 chat_history 占位符 + AgentExecutor 要挂上 memory
================================================================================
"""

import os
import dotenv

# -----------------------------------------------------------------------------
# 1) LLM：你使用的是 langchain_openai.ChatOpenAI
#    但你实际连接的是 SiliconFlow 的 base_url（兼容 OpenAI API 协议）
# -----------------------------------------------------------------------------
from langchain_openai import ChatOpenAI

# -----------------------------------------------------------------------------
# 2) 工具：TavilySearchResults（社区工具，联网搜索）
#    作用：帮 Agent 去网上查“股价、新闻、去年价格”等信息
# -----------------------------------------------------------------------------
from langchain_community.tools.tavily_search import TavilySearchResults

# -----------------------------------------------------------------------------
# 3) 工具：PythonREPL（实验包）
#    作用：让 Agent 可以执行 Python 代码做数学计算（比如涨跌幅计算）
# -----------------------------------------------------------------------------
from langchain_experimental.utilities.python import PythonREPL

# -----------------------------------------------------------------------------
# 4) Tool：把“搜索/计算”封装成 LangChain 可调用工具（Tool）
# -----------------------------------------------------------------------------
from langchain_core.tools import Tool

# -----------------------------------------------------------------------------
# 5) Prompt：新版 LangChain 推荐用 ChatPromptTemplate + MessagesPlaceholder
#    - MessagesPlaceholder 用来放“对话历史”或 “agent_scratchpad（工具思考轨迹）”
# -----------------------------------------------------------------------------
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# -----------------------------------------------------------------------------
# 6) Agent / Executor：来自 langchain_classic（兼容旧 Agents 的包）
#    - create_openai_tools_agent：创建基于 “工具调用 (Tool Calling / Function Calling)” 的 Agent
#    - AgentExecutor：负责运行 Agent（循环：思考 -> 调工具 -> 观察 -> 再思考 -> 输出）
# -----------------------------------------------------------------------------
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor

# -----------------------------------------------------------------------------
# 7) Memory：对话记忆组件（同样来自 langchain_classic）
#    - ConversationBufferMemory：保存所有历史（越聊越长）
#    - ConversationBufferWindowMemory：只保留最近 k 轮（可控长度）
# -----------------------------------------------------------------------------
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
)


# =============================================================================
# ⚙️ Step 0：环境变量准备
# =============================================================================
# dotenv.load_dotenv 会从 .env 文件读取环境变量并写入 os.environ
# override=True 表示：如果环境变量已存在，则允许 .env 覆盖它
dotenv.load_dotenv(override=True)

# NO_PROXY 常见于企业网络/代理场景：避免本地调用走代理
os.environ["NO_PROXY"] = "localhost,127.0.0.1"


# =============================================================================
# 🛠️ Step 1：打造“百宝箱”（定义多个工具）
# =============================================================================

# -----------------------------------------------------------------------------
# 工具 A：联网搜索 Search
# -----------------------------------------------------------------------------
# TavilySearchResults 是一个“工具类”，里面封装了 run() 方法用于搜索
# max_results=3 表示最多返回 3 条搜索结果（越大越“啰嗦”，也更贵更慢）
search = TavilySearchResults(max_results=3)

# Tool 是 LangChain 的“工具适配器”
# - name：工具名称（模型会看到这个名字，用于决定调用哪个工具）
# - func：真正被调用的函数（这里是 search.run）
# - description：工具描述（⚠️模型路由选择工具时最重要的依据）
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于搜索互联网上的信息，特别是股票价格和新闻。",
)

# -----------------------------------------------------------------------------
# 工具 B：数学计算 Calculator
# -----------------------------------------------------------------------------
# PythonREPL 提供一个执行 Python 代码的环境（字符串代码进去，字符串结果出来）
python_repl = PythonREPL()

calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    # ⚠️ 重要提示：PythonREPL 往往需要 print(...) 才能把结果返回给模型
    description="用于执行数学计算，例如计算百分比变化。注意：如需返回结果请 print(...)。",
)

# 将多个工具放入工具列表，Agent 会自动“看到”它们，并在需要时调用
tools = [search_tool, calc_tool]


# =============================================================================
# 🧠 Step 2：初始化大模型（LLM） + Prompt + Memory
# =============================================================================

# -----------------------------------------------------------------------------
# 2.1 初始化 LLM
# -----------------------------------------------------------------------------
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),  # 从环境变量取 key
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),  # SiliconFlow 的 OpenAI 兼容地址
    model="Qwen/Qwen3-8B",  # 你选择的模型
    temperature=0,  # 工具选择要稳定，通常设 0
)

# -----------------------------------------------------------------------------
# 2.2 初始化 Memory（对话记忆）
# -----------------------------------------------------------------------------
# 这里我们用 ConversationBufferMemory：
# - 会把所有历史对话一直保存（对话越长，context 越长）
# - return_messages=True：让 memory 输出 “消息对象列表”，而不是纯文本
# - memory_key="chat_history"：非常关键！必须和 prompt 里的变量名一致
#
# 如果你怕上下文太长，可换 ConversationBufferWindowMemory（只保留最近 k 轮）
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)
# 备选：窗口记忆（只保留最近 k 轮）
# memory = ConversationBufferWindowMemory(k=5, memory_key="chat_history", return_messages=True)

# -----------------------------------------------------------------------------
# 2.3 定义 Prompt（带对话历史 + scratchpad）
# -----------------------------------------------------------------------------
# Prompt 结构说明：
# - system：定义 Agent 角色与规则
# - chat_history：Memory 注入的位置（历史对话会插在这里）
# - human：用户本轮输入
# - agent_scratchpad：Agent 执行工具的“中间过程”占位符（工具调用轨迹）
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一位专业的金融分析特工。请利用手中的工具分步骤解决用户的问题。",
        ),
        # ✅ 让历史对话注入到 prompt（这就是“记忆生效”的关键之一）
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        # ✅ 工具调用的思考轨迹（必须保留，否则 Agent 无法正常循环工具）
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)


# =============================================================================
# 🧩 Step 3：组装 Agent + Executor（把 tools、prompt、memory 串起来）
# =============================================================================

# -----------------------------------------------------------------------------
# 3.1 创建 Agent（Tool Calling / Function Calling 模式）
# -----------------------------------------------------------------------------
# create_openai_tools_agent 会创建一个“会调用工具”的 Agent
# - 它会根据 prompt 和 tools 决定何时调用 Search / Calculator
agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

# -----------------------------------------------------------------------------
# 3.2 创建 AgentExecutor（负责运行循环）
# -----------------------------------------------------------------------------
# AgentExecutor 会负责：
#   (1) 把 input + chat_history 塞进 prompt
#   (2) 让模型思考
#   (3) 如果模型发出 tool call，就执行对应工具
#   (4) 把工具结果回填给模型
#   (5) 循环直到模型输出最终答案
#
# memory=memory：✅ 关键！把记忆组件挂上去
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,  # ✅ 记忆注入 & 自动写入
    verbose=True,  # 打开“上帝视角”日志，方便看工具调用流程
    handle_parsing_errors=True,  # 避免部分解析错误导致崩溃
)

print("\n🚀 正在唤醒双核金融特工（带记忆）...\n")


# =============================================================================
# 🚀 Step 4：运行（多轮对话示例）
# =============================================================================

# 第 1 轮问题：复合问题，需要 Search + Calculator
query1 = "特斯拉当前股价是多少？比去年上涨了百分之几？"
result1 = agent_executor.invoke({"input": query1})

print("\n" + "=" * 60)
print(f"🎯 第1轮最终分析报告：\n{result1['output']}")
print("=" * 60)

# 第 2 轮追问：不再重复问“特斯拉是谁”，而是引用上轮内容
# ✅ 如果 Memory 生效，模型会记得上一轮它查到的数据/结论，并能基于它继续回答
query2 = "把你刚才用到的去年股价来源再说一遍，并解释为什么选它。"
result2 = agent_executor.invoke({"input": query2})

print("\n" + "=" * 60)
print(f"🎯 第2轮最终分析报告：\n{result2['output']}")
print("=" * 60)


# =============================================================================
# 🔎（可选）查看 Memory 里到底存了什么
# =============================================================================
# 你可以用 memory.load_memory_variables({}) 看当前 memory 中的 chat_history
# print(memory.load_memory_variables({})["chat_history"])


🚀 正在唤醒双核金融特工...



> Entering new AgentExecutor chain...

Invoking: `Search` with `特斯拉当前股价`


[{'title': '特斯拉(TSLA)股票最新价格行情,实时走势图,股价分析预测', 'url': 'https://cn.investing.com/equities/tesla-motors', 'content': '+1.00%\n\n## 常见问题\n\n### 特斯拉(TSLA)现在的股价是多少？\n\n特斯拉现在的股价报402.51\n\n### 特斯拉的股票在哪间交易所挂牌交易？\n\n特斯拉的股票在纳斯达克股票交易所挂牌交易。\n\n### 特斯拉的股票代码是什么？\n\n特斯拉的股票代码是“TSLA。”\n\n### 特斯拉的市值是多少？\n\n截至今天，特斯拉的市值是1.51万亿。\n\n### 特斯拉的每股收益(TTM)是多少？\n\n特斯拉的每股收益(TTM)是1.08。\n\n### 特斯拉的下一财报日是哪天？\n\n特斯拉会在2026年04月28日公布下一份财报。\n\n### 特斯拉有多少名员工？\n\n特斯拉拥有134785名员工。\n\n### 特斯拉(TSLA)当前的行情是什么？\n\n截至2026年03月02日，特斯拉(TSLA)的交易价格报402.51，前一交易日收于402.51，日内波动范围介于398.11至407.12之间，52周波动范围介于214.25至498.83之间。\n\n### 特斯拉(TSLA)的分析师目标价是多少？\n\n特斯拉的12个月平均目标价为USD421.73，最高为USD600，最低为USD125。20名分析师建议买入，而8名则建议卖出，整体评级为中性，具有+4.77% 看涨的潜力。\n\n### TSLA的盘后价格是多少？\n\nTSLA的最新盘后股价为390.50，较前一交易日下跌-12.01，跌幅为-2.98%。\n\n查找股票交易商\n\n1日\n\n1周\n\n1个月\n\n6个月\n\n1年\n\n5年\n\n最大值\n\n上证指数\n\n4,182.59\n\n+19.71\n\n+0.47%\n\n富时中国A50指数\n\n14,752.29\n\n+122.61\n\n+0.84%\n\n

# ReAct

In [ ]:
import os
import dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.utilities.python import PythonREPL
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ✅ classic（legacy）agents + memory
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic import hub  # ✅ v1 迁移后，hub 在 classic 里 :contentReference[oaicite:1]{index=1}


# ==============================================================================
# ⚙️ 准备工作
# ==============================================================================
dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"

# ==============================================================================
# 🛠️ 工具
# ==============================================================================
# Tool A：搜索（输入：字符串 query）
search = TavilySearchResults(max_results=3)
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于搜索互联网上的信息，特别是股票价格和新闻。输入应为查询字符串。",
)

# Tool B：计算器（输入：字符串 python 代码）
python_repl = PythonREPL()
calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算，例如计算百分比变化。输入应为 Python 代码字符串；要看到结果请 print(...)。",
)

tools = [search_tool, calc_tool]

# ==============================================================================
# 🧠 模型
# ==============================================================================
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",
    temperature=0,
)

# ==============================================================================
# 🧠 Memory
# ==============================================================================
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)

# ==============================================================================
# 🧾 ReAct Prompt
# ==============================================================================
# ✅ 推荐：直接用 LangChain 常用的 ReAct 模板（避免自己写格式写错）
# 这个模板会引导模型输出：
# Thought / Action / Action Input / Observation ... Final Answer
react_prompt = hub.pull("hwchase17/react")

# 如果你还想在 ReAct 模板外再加“金融分析特工”的系统约束，
# 可以把它包一层 ChatPromptTemplate（可选项，保持最小改动就不用包）
# 这里我给的是“最稳妥的最小版本”：直接用 react_prompt

# ==============================================================================
# 🧩 组装 ReAct Agent + Executor（挂 memory）
# ==============================================================================
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,              # ✅ 记忆生效
    verbose=True,               # ✅ 看 Thought/Action/Observation 过程
    handle_parsing_errors=True,
    max_iterations=6,           # ✅ 防止陷入循环（ReAct 常见）
)

print("\n🚀 正在唤醒 ReAct 金融特工（带记忆）...\n")

# ==============================================================================
# 🚀 运行
# ==============================================================================
query = "特斯拉当前股价是多少？比去年上涨了百分之几？"
result = agent_executor.invoke({"input": query})

print("\n" + "=" * 50)
print(f"🎯 最终分析报告：\n{result['output']}")
print("=" * 50)

C:\Users\javis\AppData\Local\Temp\ipykernel_45544\1313616262.py:58: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(



🚀 正在唤醒 ReAct 金融特工（带记忆）...



> Entering new AgentExecutor chain...
我需要先搜索特斯拉当前的股价，然后计算它比去年上涨的百分比。
Action: Search
Action Input: 特斯拉当前股价[{'title': '特斯拉(TSLA)股票最新价格行情,实时走势图,股价分析预测', 'url': 'https://cn.investing.com/equities/tesla-motors', 'content': '+1.00%\n\n## 常见问题\n\n### 特斯拉(TSLA)现在的股价是多少？\n\n特斯拉现在的股价报402.51\n\n### 特斯拉的股票在哪间交易所挂牌交易？\n\n特斯拉的股票在纳斯达克股票交易所挂牌交易。\n\n### 特斯拉的股票代码是什么？\n\n特斯拉的股票代码是“TSLA。”\n\n### 特斯拉的市值是多少？\n\n截至今天，特斯拉的市值是1.51万亿。\n\n### 特斯拉的每股收益(TTM)是多少？\n\n特斯拉的每股收益(TTM)是1.08。\n\n### 特斯拉的下一财报日是哪天？\n\n特斯拉会在2026年04月28日公布下一份财报。\n\n### 特斯拉有多少名员工？\n\n特斯拉拥有134785名员工。\n\n### 特斯拉(TSLA)当前的行情是什么？\n\n截至2026年03月02日，特斯拉(TSLA)的交易价格报402.51，前一交易日收于402.51，日内波动范围介于398.11至407.12之间，52周波动范围介于214.25至498.83之间。\n\n### 特斯拉(TSLA)的分析师目标价是多少？\n\n特斯拉的12个月平均目标价为USD421.73，最高为USD600，最低为USD125。20名分析师建议买入，而8名则建议卖出，整体评级为中性，具有+4.77% 看涨的潜力。\n\n### TSLA的盘后价格是多少？\n\nTSLA的最新盘后股价为390.50，较前一交易日下跌-12.01，跌幅为-2.98%。\n\n查找股票交易商\n\n1日\n\n1周\n\n1个月\n\n6个月\n\n1年\n\n5年\n\n最大值\n\n上证指数\n\n4,182.59\n\n+19.71\n\n+0.47%\n\n富时中国A50

Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by ReadTimeoutError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Read timed out. (read timeout=10.0)"))'))
Content-Length: None
API Key: 
Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))"))
Content-Length: None
API Key: 
Failed t

In [4]:
import os
import dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.utilities.python import PythonREPL
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic.memory import ConversationBufferMemory


# ==============================================================================
# ⚙️ 0) 环境准备
# ==============================================================================
dotenv.load_dotenv(override=True)
os.environ["NO_PROXY"] = "localhost,127.0.0.1"


# ==============================================================================
# 🧰 1) 工具箱：Search + Calculator
# ==============================================================================
# 工具 A：联网搜索（Tavily）
search = TavilySearchResults(max_results=3)
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于搜索互联网上的信息，特别是股票价格和新闻。输入为查询字符串。",
)

# 工具 B：数学计算（Python REPL）
python_repl = PythonREPL()
calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算（Python）。输入为 Python 代码字符串；如需返回值请 print(...)。",
)

tools = [search_tool, calc_tool]


# ==============================================================================
# 🧠 2) 初始化大模型
# ==============================================================================
llm = ChatOpenAI(
    api_key=os.getenv("SILICON_FLOW_API_KEY"),
    base_url=os.getenv("SILICON_FLOW_BASE_URL"),
    model="Qwen/Qwen3-8B",
    temperature=0,
)


# ==============================================================================
# 🧠 3) Memory：对话记忆（必须对齐 prompt 的 {chat_history}）
# ==============================================================================
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)


# ==============================================================================
# 🧾 4) 官方 ReAct Prompt（直接写入，不依赖 hub）
# ==============================================================================
OFFICIAL_REACT_PROMPT = """Assistant is a large language model trained by OpenAI.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, Assistant is able to generate human-like text based on the input it receives, allowing it to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range of topics.

Overall, Assistant is a powerful tool that can help with a wide range of tasks and provide valuable insights and information on a wide range of topics. Whether you need help with a specific question or just want to have a conversation about a particular topic, Assistant is here to assist.

TOOLS:
------

Assistant has access to the following tools:

{tools}

To use a tool, please use the following format:

Thought: Do I need to use a tool? Yes
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action

When you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:

Thought: Do I need to use a tool? No
Final Answer: [your response here]


Begin!

Previous conversation history:
{chat_history}

New input: {input}
{agent_scratchpad}
"""

react_prompt = ChatPromptTemplate.from_template(OFFICIAL_REACT_PROMPT)


# ==============================================================================
# 🧩 5) 创建 ReAct Agent（文本解析 Thought/Action/Observation）
# ==============================================================================
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt,
)

# ==============================================================================
# 🚀 6) Executor：运行 Agent（挂上 Memory）
# ==============================================================================
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,  # ✅ 记忆功能生效
    verbose=True,  # ✅ 打印 Thought/Action/Observation 过程
    handle_parsing_errors=True,
    max_iterations=6,  # ✅ 防止陷入循环
)

print("\n🚀 正在唤醒 ReAct 金融特工（官方 Prompt + Memory）...\n")


# ==============================================================================
# ▶️ 7) 调用示例（多轮对话）
# ==============================================================================
query1 = "特斯拉当前股价是多少？比去年上涨了百分之几？"
res1 = agent_executor.invoke({"input": query1})
print("\n" + "=" * 60)
print("🎯 第1轮输出：\n", res1["output"])
print("=" * 60)

query2 = "你刚才使用了哪些工具？每个工具分别做了什么？"
res2 = agent_executor.invoke({"input": query2})
print("\n" + "=" * 60)
print("🎯 第2轮输出：\n", res2["output"])
print("=" * 60)


🚀 正在唤醒 ReAct 金融特工（官方 Prompt + Memory）...



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Search
Action Input: 特斯拉当前股价和去年的股价变化
[{'title': 'Tesla - 16 Year Stock Price History | TSLA', 'url': 'https://www.macrotrends.net/stocks/charts/TSLA/tesla/stock-price-history', 'content': 'The all-time high Tesla stock closing price was 489.88 on December 16, 2025.\n The Tesla 52-week high stock price is 498.83, which is 19.5% above the current share price.\n The Tesla 52-week low stock price is 214.25, which is 48.7% below the current share price.\n The average Tesla stock price for the last 52 weeks is 363.88.\n\n| Tesla Historical Annual Stock Price Data | [...] | Year | Average Stock Price | Year Open | Year High | Year Low | Year Close | Annual % Change |\n| 2026 | 426.3622 | 438.0700 | 451.6700 | 397.2100 | 417.3100 | -7.21% |\n| 2025 | 356.9708 | 379.2800 | 489.8800 | 221.8600 | 449.7200 | 11.36% |\n| 2024 | 230.6150 | 248.4200 | 479.8600 | 142.0500 | 